<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

DATA = "data/raw/content_refresh_anonymized.csv"
if not Path(DATA).exists():                       # in Colab: fetch the repo, the data ships inside it
    if not Path("flyrank-ml-internship").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Eng7ouda06/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")

df = pd.read_csv(DATA)
FLOOR = 1000                                      # impression floor for reading a CTR
BANDS = ["top_3", "page_1", "striking", "page_3_5", "deep"]

df["ctr"] = df.clicks_90d / df.impressions_90d * 100                       # CTR in %, rebuilt from raw counts
pos = df.avg_position.where(df.avg_position > 0)                           # avg_position == 0 means "no data"
df["band"] = pd.cut(pos, [0, 3, 10, 20, 50, np.inf], right=False, labels=BANDS)
print(len(df), "pages |", (df.avg_position == 0).sum(), "have no position data")

30000 pages | 1205 have no position data


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

What I see. Everything that is counted is heavy-tailed: impressions have a median of 731 but a mean of 5,200 (skew 11.4), clicks a median of 1 and a mean of 16, keyword volume a median of 10 and a max of 74,000. The top 1% of pages hold 24.9% of impressions and 35.0% of clicks, and 44% of pages have zero clicks, so I use pooled rates, buckets and rank correlations instead of plain averages. days_since_last_update has almost no tail (99% of pages are 106 days or less), which section 3 explains. Keyword volume is missing for 100% of feedly articles and 1.4% of keyword articles, so I never fill it with 0.

In [ ]:
key = ["impressions_90d", "clicks_90d", "sessions_90d", "search_volume", "content_age_days", "days_since_last_update"]
d = df[key].agg(["median", "mean"]).T
d["p90"], d["p99"], d["max"], d["skew"] = df[key].quantile(.9), df[key].quantile(.99), df[key].max(), df[key].skew()
print(d.round(1))

imp = df.impressions_90d.sort_values(ascending=False)
clk = df.clicks_90d.sort_values(ascending=False)
print(f"\ntop 1% of pages hold {imp.head(300).sum() / imp.sum():.1%} of impressions and {clk.head(300).sum() / clk.sum():.1%} of clicks")
print(f"{(df.clicks_90d == 0).mean():.0%} of pages have zero clicks")
print("\nshare of search_volume missing, by content type:")
print(df.groupby("content_type").search_volume.apply(lambda s: s.isna().mean()).round(3))

                        median    mean      p90      p99       max  skew
impressions_90d          731.0  5200.4  12136.4  73505.8  517715.0  11.4
clicks_90d                 1.0    16.1     32.0    253.0    4178.0  18.3
sessions_90d               7.0    37.1     88.0    451.0    4345.0  12.1
search_volume             10.0   158.9    110.0   2900.0   74000.0  26.0
content_age_days         236.0   256.2    463.0    537.0     564.0   0.5
days_since_last_update    20.0    46.1    104.0    106.0     373.0   1.2

top 1% of pages hold 24.9% of impressions and 35.0% of clicks
44% of pages have zero clicks

share of search_volume missing, by content type:
content_type
comparison article    0.000
feedly article        1.000
keyword article       0.014
Name: search_volume, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Each test states a claim, shows a table with n (number of pages), and ends with one verdict word. Rates are pooled (total clicks ÷ total impressions), never an average of page CTRs.

Signal 1: CTR falls as position gets worse (pages with 1,000+ impressions). Linked flag: the CTR-fix logic.

Verdict 1: MIXED. Coarsely true, not band by band: CTR is 0.487% in the top-3 band and 0.036% in the deep band (about 13.5×). But page 1 (0.351%) and page 2 (0.354%) are tied, and the top-3 median (0.226%) is below page 1's (0.245%). In practice: adjust CTR for position only coarsely, and do not treat page 1 vs page 2 as different.

In [ ]:
seen = df[df.impressions_90d >= FLOOR]
A = seen.groupby("band", observed=True).agg(n=("content_id", "size"), impressions=("impressions_90d", "sum"),
                                            clicks=("clicks_90d", "sum"), median_ctr=("ctr", "median"))
A["pooled_ctr"] = A.clicks / A.impressions * 100
print(A[["n", "pooled_ctr", "median_ctr"]])

if A.pooled_ctr.is_monotonic_decreasing and A.median_ctr.is_monotonic_decreasing:
    V1 = "CONFIRMED"
elif A.pooled_ctr.iloc[0] > 2 * A.pooled_ctr.iloc[-1]:
    V1 = "MIXED"                                   # right direction overall, not band by band
elif A.pooled_ctr.iloc[0] < A.pooled_ctr.iloc[-1]:
    V1 = "OPPOSITE"
else:
    V1 = "FALSE"
print("VERDICT 1:", V1)

             n  pooled_ctr  median_ctr
band                                  
top_3      373    0.486667    0.226180
page_1    6145    0.350864    0.244584
striking  3470    0.354171    0.185782
page_3_5  3334    0.156103    0.095064
deep       190    0.036129    0.000000
VERDICT 1: MIXED


Signal 2: CTR is only readable with enough impressions (pages with position under 20).

Verdict 2: CONFIRMED. The share of pages with zero clicks falls at every step: 87% under 50 impressions, 54% at 250–499, 10% at 1,000–4,999, under 1% above 5,000. At a typical CTR of 0.359%, a page with 500 impressions still has a 16.6% chance of zero clicks by luck; at 1,000 it is 2.8%. In practice: below about 1,000 impressions, "low CTR" mostly means "low volume".

In [ ]:
low = df[(df.avg_position > 0) & (df.avg_position < 20)].copy()
low["volume"] = pd.cut(low.impressions_90d, [1, 50, 100, 250, 500, 1000, 5000, np.inf], right=False,
                       labels=["1-49", "50-99", "100-249", "250-499", "500-999", "1,000-4,999", "5,000+"])
B = low.groupby("volume", observed=True).agg(n=("content_id", "size"),
                                             share_with_0_clicks=("clicks_90d", lambda s: (s == 0).mean()))
print(B)

big = low[low.impressions_90d >= FLOOR]
typical = big.clicks_90d.sum() / big.impressions_90d.sum()
print(f"typical CTR {typical:.3%}: chance of 0 clicks by pure luck at 500 impressions = {np.exp(-500 * typical):.1%}, at 1,000 = {np.exp(-1000 * typical):.1%}")

V2 = "CONFIRMED" if B.share_with_0_clicks.is_monotonic_decreasing else "MIXED"
print("VERDICT 2:", V2)

                n  share_with_0_clicks
volume                                
1-49         4146             0.872648
50-99        1006             0.798211
100-249      1526             0.658585
250-499      1527             0.540275
500-999      2006             0.328016
1,000-4,999  5302             0.098265
5,000+       4686             0.006829
typical CTR 0.359%: chance of 0 clicks by pure luck at 500 impressions = 16.6%, at 1,000 = 2.8%
VERDICT 2: CONFIRMED


Signal 3: pages targeting higher-volume keywords get more impressions. Linked flag: the volume assumption behind quick-win.

Verdict 3: FALSE. Median impressions do not rise with keyword volume (998, 834, 935, 843, 780 and 846 across the buckets). Spearman is −0.029, and Pearson is +0.001 on raw values but −0.026 after a log transform: both near zero, with the sign flipping. Every bucket has at least 560 pages. In practice: here the keyword-volume estimate does not tell you which pages get impressions, so I would not rank pages on it. One caution: it estimates the target keyword, while a page's impressions come from many queries.

In [ ]:
kw = df.dropna(subset=["search_volume"]).copy()          # feedly articles have no keyword data: left out, never filled with 0
kw["bucket"] = pd.cut(kw.search_volume, [-1, 0, 10, 50, 200, 1000, np.inf],
                      labels=["0", "1-10", "11-50", "51-200", "201-1,000", "1,000+"])
C = kw.groupby("bucket", observed=True).agg(n=("content_id", "size"), median_impressions=("impressions_90d", "median"))
print(C)

rho = kw.search_volume.rank().corr(kw.impressions_90d.rank())             # Spearman = Pearson on ranks
r_raw = kw.search_volume.corr(kw.impressions_90d)
r_log = np.log1p(kw.search_volume).corr(np.log1p(kw.impressions_90d))
print(f"Spearman {rho:+.3f} | Pearson on raw values {r_raw:+.3f} | Pearson after log {r_log:+.3f}")

V3 = "FALSE" if abs(rho) < 0.10 else ("CONFIRMED" if rho >= 0.30 else ("OPPOSITE" if rho <= -0.30 else "MIXED"))
print("VERDICT 3:", V3)

               n  median_impressions
bucket                              
0          11081               998.0
1-10        7311               834.0
11-50       4989               935.0
51-200      2081               843.0
201-1,000   1510               779.5
1,000+       560               845.5
Spearman -0.029 | Pearson on raw values +0.001 | Pearson after log -0.026
VERDICT 3: FALSE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Assumption behind the refresh flags: pages that have not been updated for a while are the ones that decline. The product's flags are not shipped in this data, so I test the assumption itself. Outcome: trend_direction == "down" (impressions in the last 30 days fell more than 20% vs the 30 before). That is the starter label, so it is used here as an outcome only, never as a feature. Compare: stale (91+ days since update) vs fresh (30 days or fewer), first overall, then within content-age groups, because older pages differ in both staleness and decline.

In [ ]:
d = df.copy()
d["down"] = d.trend_direction == "down"
d["group"] = np.select([d.days_since_last_update <= 30, d.days_since_last_update >= 91], ["fresh", "stale"], default="mid")
d["age"] = pd.cut(d.content_age_days, [0, 180, 365, np.inf], labels=["<=180 d", "181-365 d", "365+ d"])

print("pages at the flag's own threshold (180+ days since update AND 500+ impressions):", ((d.days_since_last_update >= 180) & (d.impressions_90d >= 500)).sum())
top4 = d.days_since_last_update.value_counts().head(4)
print(f"most common days_since_last_update values: {top4.to_dict()} = {top4.sum() / len(d):.0%} of pages")
print(f"share of pages that are 'down': {d.down.mean():.0%}\n")

def gap(x):
    """Share of 'down' pages, stale minus fresh, in percentage points, inside each content-age group."""
    t = x[x.group != "mid"].groupby(["age", "group"], observed=True).down.agg(["size", "mean"]).unstack("group")
    return pd.DataFrame({"n_stale": t[("size", "stale")], "n_fresh": t[("size", "fresh")], "down_stale": t[("mean", "stale")],
                         "down_fresh": t[("mean", "fresh")], "gap_pp": (t[("mean", "stale")] - t[("mean", "fresh")]) * 100})

visible = d[d.impressions_90d >= 500]
for name, x in [("ALL PAGES", d), ("PAGES WITH 500+ IMPRESSIONS", visible)]:
    raw = x[x.group != "mid"].groupby("group").down.mean()
    print(f"{name}: down share stale {raw['stale']:.1%} vs fresh {raw['fresh']:.1%}  (raw gap {100 * (raw['stale'] - raw['fresh']):+.1f} pp)")
    print(gap(x).round(3), "\n")

V4 = "CONFIRMED" if (gap(d).gap_pp >= 5).all() and (gap(visible).gap_pp >= 5).all() else ("MIXED" if (gap(d).gap_pp >= 5).all() else "FALSE")
print("VERDICT 4 (flag-linked):", V4)

pages at the flag's own threshold (180+ days since update AND 500+ impressions): 17
most common days_since_last_update values: {20: 11573, 104: 8773, 22: 3564, 8: 1929} = 86% of pages
share of pages that are 'down': 54%

ALL PAGES: down share stale 60.8% vs fresh 51.1%  (raw gap +9.7 pp)
           n_stale  n_fresh  down_stale  down_fresh  gap_pp
age                                                        
<=180 d       2204     9996       0.701       0.611   9.001
181-365 d     6601     4677       0.586       0.412  17.338
365+ d         540     5807       0.506       0.419   8.675 

PAGES WITH 500+ IMPRESSIONS: down share stale 61.6% vs fresh 58.3%  (raw gap +3.4 pp)
           n_stale  n_fresh  down_stale  down_fresh  gap_pp
age                                                        
<=180 d       1653     5150       0.725       0.664   6.086
181-365 d     4545     1420       0.589       0.637  -4.833
365+ d         377     3493       0.472       0.440   3.184 

VERDICT 4 (flag-linke

Verdict 4: MIXED.

The flag's own threshold can't be tested: only 17 pages have 180+ days since update and 500+ impressions, far under the 50-page floor.
The staleness variable looks like batch updates: 86% of pages sit on one of four values (20, 104, 22 or 8 days), so "stale" mostly says which update batch a page was in.
Across all pages, stale does go with decline: 60.8% of stale pages are "down" vs 51.1% of fresh ones (+9.7 pp), and the gap holds in all three content-age groups (+9.0, +17.3, +8.7 pp).
Where a flag would act, it mostly disappears: with 500+ impressions the raw gap is +3.4 pp, and within age groups it is +6.1, −4.8 and +3.2 pp, so the sign flips for pages 181–365 days old.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Don't rank pages by keyword search volume: here it tells you nothing about which pages earn impressions. Judge CTR only against a coarse position group and only past roughly 1,000 impressions. Don't treat "days since update" alone as a reason to refresh a page that already gets real traffic, because among visible pages it barely separates the ones that decline; these are observed, decision-support findings, not proof that a refresh or rewrite recovers clicks.

In [ ]:
summary = pd.DataFrame({"test": ["1. CTR falls as position worsens", "2. CTR needs volume to be readable",
                                 "3. keyword volume -> impressions", "4. stale pages decline more (flag-linked)"],
                        "verdict": [V1, V2, V3, V4]})
print(summary.to_string(index=False))

os.makedirs("work/outputs", exist_ok=True)
json.dump(dict(zip(["signal1_position", "signal2_volume", "signal3_keyword_volume", "signal4_staleness"], [V1, V2, V3, V4])),
          open("work/outputs/w04_signal_audit.json", "w"), indent=2)

                                     test   verdict
         1. CTR falls as position worsens     MIXED
       2. CTR needs volume to be readable CONFIRMED
         3. keyword volume -> impressions     FALSE
4. stale pages decline more (flag-linked)     MIXED
